<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 12 — Things and Connections

A table describes things: one row each, with their properties. Some questions
are not about the things but about what is **between** them. Who knows whom,
which page links to which, what can be reached from where. The structure for
that is a **graph**, and this session is about building one and searching it.

Six ideas, one part each:

| Part | Idea | In one sentence |
|---|---|---|
| 1 | Nodes and edges | A graph is a set of things and a set of connections between pairs of them. |
| 2 | Writing a graph down | An edge list is easy to store; an adjacency list is quick to ask. |
| 3 | Breadth-first search | Explore in rings: everything one step away, then two, then three. |
| 4 | The path | Remember where each node was reached from, and walk back. |
| 5 | What belongs together | Reachability, and the separate parts of a network. |
| 6 | When edges have lengths | Fewest steps is not shortest distance, and needs another search. |

**How this session's notebooks fit together**

| Notebook | What it is |
|---|---|
| `01_Things_and_Connections` (this one) | The ideas, on graphs you can draw |
| `02_Word_Ladders_Example` | One graph built and explored: 2,442 words, joined when one letter apart |
| `03_Graphs_Reference` | A lookup guide to keep open while you do Homework 11 |

**What is not repeated here.** Dictionaries and sets are Sessions 3 and 4, and
why a lookup in them takes one step is Session 11. This session uses both
constantly.

**Before you start**

- Set the kernel (upper right corner) to **Conda Python 3.12**.
- Run the cells in order with **Shift + Enter**. Before each cell, guess what
  it will show.

In [ ]:
from collections import deque

import matplotlib.pyplot as plt
import networkx as nx

---

## Part 1 — Nodes and edges

Seven people share a flat-hunting group chat. Some of them know each other
from before.

In [ ]:
friendships = [("Ana", "Ben"), ("Ana", "Chi"), ("Ben", "Chi"), ("Chi", "Dev"),
               ("Dev", "Eli"), ("Eli", "Fay"), ("Dev", "Fay"), ("Gus", "Hal")]

# Where each person is drawn.  The positions mean nothing; they only make the
# picture the same every time.
PLACES = {"Ana": (0, 2), "Ben": (0, 0), "Chi": (1.5, 1), "Dev": (3, 1),
          "Eli": (4.5, 2), "Fay": (4.5, 0), "Gus": (6, 2), "Hal": (6, 0)}


def draw(edges, title, directed=False, labels=None):
    """Draw a small graph with networkx.  Only for looking at it."""
    picture = nx.DiGraph() if directed else nx.Graph()
    picture.add_nodes_from(PLACES)
    picture.add_edges_from(edges)
    fig, ax = plt.subplots(figsize=(8, 2.8))
    nx.draw_networkx(picture, pos=PLACES, ax=ax, node_color="lightsteelblue",
                     node_size=1100, font_size=10)
    if labels:
        nx.draw_networkx_edge_labels(picture, pos=PLACES, edge_labels=labels, ax=ax)
    ax.set_title(title)
    ax.axis("off")
    fig.tight_layout()


draw(friendships, "Who knows whom")

That picture is a **graph**. It has two kinds of thing in it, and nothing
else:

| | Called | Here |
|---|---|---|
| the dots | **nodes** (or vertices) | people |
| the lines | **edges** | "these two know each other" |

Where a node is drawn means nothing. Only *which nodes are joined* matters.
Redraw it with everyone in a circle and it is the same graph.

Graphs are everywhere once you look:

| Nodes | Edges | A question about it |
|---|---|---|
| airports | routes | how do I get from Atlanta to Sydney? |
| web pages | links | which pages can be reached from the home page? |
| people | friendships | how many introductions separate two strangers? |
| courses | "is required for" | in what order can I take them? |
| words | "differ by one letter" | can COLD be turned into WARM? |

Some words for what you can see in the picture:

- Ana's **neighbours** are Ben and Chi: the nodes one edge away.
- Dev has **degree** 3: three edges meet there.
- Ana, Chi, Dev, Eli is a **path**: a sequence of nodes, each joined to the
  next. It has three edges; people say it is three **hops** long.
- Gus and Hal are cut off from the others: the graph has two separate
  **parts**.

### Two ways edges can differ

"Knows" goes both ways. "Follows" does not: I can follow you without you
following me. An edge with a direction is drawn as an arrow, and the graph is
called **directed**.

In [ ]:
follows = [("Ana", "Ben"), ("Ben", "Ana"), ("Ana", "Chi"), ("Chi", "Dev"),
           ("Dev", "Eli"), ("Fay", "Eli"), ("Fay", "Dev"), ("Gus", "Hal")]

draw(follows, "Who follows whom", directed=True)

Ana and Ben follow each other: two arrows. Chi follows Dev, and Dev does not
follow back. Whether a path exists now depends on the direction: from Ana you
can get to Eli, and from Eli you can get nowhere.

An edge can also carry a number: a distance, a cost, a travel time. That
number is its **weight**, and the graph is **weighted**. Part 6 comes back to
it.

| | Undirected | Directed |
|---|---|---|
| **Unweighted** | friendships | who follows whom, web links |
| **Weighted** | roads with their lengths | flights with their durations |

**✏️ Try it:** In the directed picture, who can Fay reach by following arrows? Who can reach Fay?

---

## Part 2 — Writing a graph down

A picture is for people. A program needs the graph as data, and there are two
usual forms.

**An edge list**: just the edges, one after another. `friendships` above is
one. It is how graphs are stored in files, one edge per row.

It is a poor form to *ask*. Who does Dev know?

In [ ]:
dev_knows = []
for a, b in friendships:
    if a == "Dev":
        dev_knows.append(b)
    if b == "Dev":
        dev_knows.append(a)
print(dev_knows)

Every edge had to be looked at: a linear search (Session 11). A search through
the graph asks "who are this node's neighbours?" over and over, so this would
be a linear search inside a loop.

**An adjacency list**: for every node, its neighbours, in a dictionary.

In [ ]:
def build_graph(edges, directed=False):
    """Return {node: set of neighbours}, with every node as a key."""
    graph = {}
    for a, b in edges:
        graph.setdefault(a, set())
        graph.setdefault(b, set())          # b too, even if nothing leaves it
        graph[a].add(b)
        if not directed:
            graph[b].add(a)
    return graph


friends = build_graph(friendships)
for person in sorted(friends):
    print(person, sorted(friends[person]))

Now "who does Dev know?" is `friends["Dev"]`: one dictionary lookup, however
large the graph. "Do Ana and Chi know each other?" is `"Chi" in
friends["Ana"]`, one more.

Each friendship is stored twice, once from each end. That is what "undirected"
means in this form.

Look at the second `setdefault`. It makes sure a node is in the dictionary
even when it only ever appears as the *second* name of an edge. In a directed
graph that happens all the time:

In [ ]:
following = build_graph(follows, directed=True)
for person in sorted(following):
    print(person, sorted(following[person]))

Eli and Hal follow nobody. They are still nodes, with an empty set of
neighbours. Leave that `setdefault` out and they vanish from the dictionary,
and the first function that asks for `following["Eli"]` stops with a
`KeyError`. It is the most common bug in a first graph program.

(A third form, a table with a row and a column per node and a 1 where there is
an edge, is called an adjacency matrix. For a network of 3,000 airports it has
nine million cells, nearly all of them 0. Real networks are sparse, and the
dictionary stores only what is there.)

**✏️ Try it:** Using `friends`: what is the degree of each person? Who has the highest? One line with a dictionary comprehension.

---

## Part 3 — Breadth-first search

Ana wants an introduction to Fay. How many introductions does it take? In
graph terms: what is the smallest number of edges on a path from Ana to Fay?

Do it by hand, in rings around Ana.

In [ ]:
ring = ["Ana"]
found = {"Ana"}
steps = 0
while ring:
    print(steps, "steps from Ana:", sorted(ring))
    next_ring = []
    for person in ring:
        for neighbour in sorted(friends[person]):
            if neighbour not in found:
                found.add(neighbour)
                next_ring.append(neighbour)
    ring = next_ring
    steps += 1

Everyone one step away is found before anyone two steps away, and so on
outward. That is why the first time a node is found is by a shortest route: a
ring further out cannot get there sooner.

Gus and Hal never appear. They cannot be reached at all.

The usual way to write this uses one **queue** in place of the ring-by-ring
lists. A queue is a line at a counter: you join at the back and are served
from the front. `collections.deque` is one.

In [ ]:
def hop_distances(graph, start):
    """Return {node: fewest steps from start} for every reachable node."""
    hops = {start: 0}                    # found so far, and how far away
    queue = deque([start])               # found, neighbours not yet looked at
    while queue:
        node = queue.popleft()           # from the FRONT
        for neighbour in sorted(graph[node]):
            if neighbour not in hops:    # not seen before
                hops[neighbour] = hops[node] + 1
                queue.append(neighbour)  # to the BACK
    return hops


hop_distances(friends, "Ana")

This is **breadth-first search**. Follow it once with the queue printed:

In [ ]:
hops = {"Ana": 0}
queue = deque(["Ana"])
while queue:
    print("queue:", list(queue))
    node = queue.popleft()
    for neighbour in sorted(friends[node]):
        if neighbour not in hops:
            hops[neighbour] = hops[node] + 1
            queue.append(neighbour)

The queue always holds nodes of at most two neighbouring rings, nearer ones in
front. That is the ring order, kept by the queue without anyone managing it.

Three details decide whether it works.

**Take from the front.** Take from the back instead, and the search runs down
one path as far as it goes before trying another. That is a different search
(depth-first). It finds the same nodes with the wrong distances. Six people sit
around a table, each knowing the two beside them:

In [ ]:
def wrong_distances(graph, start):
    hops = {start: 0}
    queue = deque([start])
    while queue:
        node = queue.pop()               # from the BACK
        for neighbour in sorted(graph[node]):
            if neighbour not in hops:
                hops[neighbour] = hops[node] + 1
                queue.append(neighbour)
    return hops


table = build_graph([("P", "Q"), ("Q", "R"), ("R", "S"), ("S", "T"), ("T", "U"), ("U", "P")])

print("front:", hop_distances(table, "P"))
print("back: ", wrong_distances(table, "P"))

From the front: P's neighbours at 1, the next two at 2, and S, opposite, at 3.
From the back: the search went all the way round the table in one direction
and reports R as four seats away. R sits two seats from P.

**Skip what has been seen.** Without `if neighbour not in hops`, the search
goes Ana, Ben, Ana, Ben, ... for ever. Any graph with a cycle traps it.

**Note a node when it joins the queue**, not when it leaves. Otherwise a node
with five neighbours already in the queue is added five times.

The cost: each node joins the queue once and each edge is looked at once (or
twice, from both ends). The work grows with the size of the graph, linearly.
For a search that can answer "how far is everything from here?", that is as
cheap as it could be.

**✏️ Try it:** Run `hop_distances(following, "Ana")` on the directed graph. Then from "Eli". Explain the second result.

---

## Part 4 — The path

Three steps from Ana to Fay. Through whom?

The search knows and did not say. Each time it finds a new node, it finds it
*from* the node it is standing on. Record that one fact per node.

In [ ]:
def shortest_path(graph, start, target):
    """Return a path with the fewest steps, or None."""
    if start not in graph or target not in graph:
        return None
    came_from = {start: None}            # the start was reached from nowhere
    queue = deque([start])
    while queue:
        node = queue.popleft()
        for neighbour in sorted(graph[node]):
            if neighbour not in came_from:
                came_from[neighbour] = node
                queue.append(neighbour)

    if target not in came_from:
        return None                      # never reached

    path = []
    node = target
    while node is not None:              # walk the notes back to the start
        path.append(node)
        node = came_from[node]
    path.reverse()
    return path


print(shortest_path(friends, "Ana", "Fay"))
print(shortest_path(friends, "Ana", "Gus"))
print(shortest_path(friends, "Ana", "Ana"))

`came_from` does two jobs: it is the record of which nodes have been seen, and
it is one note per node saying "I was first reached from here". Following the
notes from the target gives the path backwards, so it is reversed at the end.

**Shortest paths are not unique.** From Ana to Fay there is exactly one path
of three steps here. Add one friendship and there are two:

In [ ]:
more = build_graph(friendships + [("Chi", "Eli")])
print(shortest_path(more, "Ana", "Fay"))
print("also three steps: ['Ana', 'Chi', 'Eli', 'Fay']")

Which one the function returns depends on the order in which it looks at
neighbours: alphabetical here, so Dev before Eli. Both are right. Whenever you
check a shortest path, check that it is a real path and that its *length* is
right, not that it is one particular path.

**✏️ Try it:** Print the `came_from` dictionary for a search from "Ana" (copy the loop out of the function). Read off the path to "Eli" by hand.

---

## Part 5 — What belongs together

`hop_distances` already answers "what can be reached from here?": its keys.

In [ ]:
print(sorted(hop_distances(friends, "Ana")))
print(sorted(hop_distances(friends, "Gus")))

Two groups with nothing between them. Each is a **connected component**: a set
of nodes that can all reach each other, with no edge leading out. Finding all
of them is a loop around the search: take any node not yet placed, collect
what it can reach, repeat.

In [ ]:
def components(graph):
    """Return the connected parts of an undirected graph, largest first."""
    seen = set()
    parts = []
    for node in graph:
        if node not in seen:
            part = set(hop_distances(graph, node))
            seen |= part                 # add the whole part to seen
            parts.append(part)
    parts.sort(key=lambda part: (-len(part), min(part)))
    return parts


components(friends)

On real networks this is one of the first things to look at. A social network
usually has one giant component with nearly everyone in it and a dust of tiny
ones. A road network that falls into two components has a bridge missing, or
an error in the data.

For a **directed** graph "can reach each other" needs care: in the "follows"
graph Ana can reach Eli and Eli can reach nobody. The simple question, which
nodes hang together at all, is answered by ignoring the directions first.

---

## Part 6 — When edges have lengths

Four towns and the roads between them, with distances in kilometres.

In [ ]:
PLACES = {"Alba": (0, 1), "Brig": (2, 2), "Cove": (2, 0), "Dale": (4, 1)}
roads = {("Alba", "Dale"): 90, ("Alba", "Brig"): 20, ("Brig", "Cove"): 15,
         ("Cove", "Dale"): 25, ("Alba", "Cove"): 50}

draw(list(roads), "Roads, with their lengths in km", labels=roads)

From Alba to Dale. Breadth-first search counts roads:

In [ ]:
towns = build_graph(list(roads))
print(shortest_path(towns, "Alba", "Dale"))

One road, the direct one, and it is 90 km long. The route through Brig and
Cove takes three roads and 20 + 15 + 25 = 60 km. **Fewest steps is not
shortest distance.** Breadth-first search is not wrong; it answers the
question of how many edges, and here the question was how many kilometres.

To find the shortest route by distance, the search has to change in one
respect. Breadth-first search deals with nodes in the order they were found.
The weighted search deals with them in order of their **distance from the
start**: always the nearest node not yet finished. That is **Dijkstra's
algorithm**. The queue becomes a *priority queue*, which hands back the
smallest item, not the oldest. Python's is `heapq`.

In [ ]:
import heapq


def nearest_first(graph, lengths, start, target):
    """Dijkstra: the path with the smallest total length, and that length."""
    best = {start: 0}
    came_from = {start: None}
    queue = [(0, start)]                         # (distance so far, node)
    done = set()
    while queue:
        so_far, node = heapq.heappop(queue)      # the NEAREST, not the oldest
        if node in done:
            continue
        done.add(node)
        for neighbour in sorted(graph[node]):
            length = lengths.get((node, neighbour)) or lengths[(neighbour, node)]
            if neighbour not in best or so_far + length < best[neighbour]:
                best[neighbour] = so_far + length      # a shorter way: note it
                came_from[neighbour] = node
                heapq.heappush(queue, (so_far + length, neighbour))
    path = []
    node = target
    while node is not None:
        path.append(node)
        node = came_from[node]
    return path[::-1], best[target]


nearest_first(towns, roads, "Alba", "Dale")

Compare it with `shortest_path` line by line. The differences are few:

| Breadth-first search | Dijkstra |
|---|---|
| a queue: oldest first | a priority queue: nearest first |
| a node's distance is fixed when it is first found | it may be improved until the node is taken from the queue |
| counts edges | adds up weights |

With every edge of length 1, Dijkstra *is* breadth-first search.

You do not write Dijkstra in Homework 11; it is given, and you read it. What
you need from this part is the distinction: before searching, ask what
"shortest" is supposed to mean.

### The library

Everything in this notebook exists in **networkx**. Your functions and its
functions, side by side:

In [ ]:
G = nx.Graph()
G.add_edges_from(friendships)

print(nx.shortest_path(G, "Ana", "Fay"), shortest_path(friends, "Ana", "Fay"))
print(dict(nx.single_source_shortest_path_length(G, "Ana")) == hop_distances(friends, "Ana"))
print(sorted(nx.connected_components(G), key=len, reverse=True) == components(friends))

R = nx.Graph()
R.add_weighted_edges_from((a, b, km) for (a, b), km in roads.items())
print(nx.shortest_path(R, "Alba", "Dale"), nx.shortest_path(R, "Alba", "Dale", weight="weight"))

The last line is the whole of Part 6: the same function gives two answers,
depending on whether it is told that the edges have weights.

In real work, use the library. It is tested, fast, and has hundreds of
algorithms. You write the search yourself once, in Homework 11, so that you
know what the library does, what it costs, and which question each of its
functions answers.

**✏️ Try it:** Change the Alba–Dale road to 55 km. Which route is shortest now? And at 60 km?

---

## Exercises

Use `friendships`, `friends` and the functions defined above.

### 🧩 Exercise 1 — Build and ask

A course catalogue: `("Intro", "Data")`, `("Intro", "Stats")`,
`("Data", "ML")`, `("Stats", "ML")`, `("ML", "Capstone")`, `("Ethics",
"Capstone")`, where `(a, b)` means "a is required for b". Build it as a
directed graph. Which courses require nothing first (no edge leads *to*
them)? Which course is required by nothing?

### 🧩 Exercise 2 — Rings

On paper, do a breadth-first search from "Dev" in the friendship graph: write
the queue after each step and the distance of each person. Then check with
`hop_distances(friends, "Dev")`.

### 🧩 Exercise 3 — Farthest apart

Which two people in the large group are farthest apart, and how many steps?
Call `hop_distances` from every person and keep the largest value. (That
number is called the *diameter* of the network.)

### 🧩 Exercise 4 — One friendship more

Add the single friendship that joins the two components, and check with
`components`. Then: which single friendship, if it ended, would split the
large group in two? (Such an edge is called a *bridge*. Find it by trying each
edge in turn.)

---

## Summary

| Idea | The habit | The tool |
|---|---|---|
| Nodes and edges | Say what a node is and what an edge means | a sketch |
| Writing a graph down | Store edges; ask an adjacency list; every node a key | `{node: neighbours}` |
| Breadth-first search | A queue, taken from the front; skip what was seen | `deque`, a dictionary of what was found |
| The path | One "came from" note per node, walked backwards | `came_from`, `reverse` |
| What belongs together | Look at the components before anything else | a loop around the search |
| Edges with lengths | Ask what "shortest" means before searching | Dijkstra; `networkx` |

### Where each idea appears in Homework 11

| Idea | Homework 11 |
|---|---|
| An edge list from a file | Phase 1, `read_edges` |
| An adjacency list with every node present | Phase 2, `build_graph` |
| Neighbours and degree | Phase 3, `neighbors`, `degree`, `has_edge`, `hubs` |
| Breadth-first search | Phase 4, `hop_distances` |
| `came_from`, walked backwards | Phase 5, `shortest_path` |
| Reachability and components | Phase 6, `reachable`, `components` |
| Fewest steps against shortest distance | Phase 7, `path_length`, `compare_routes` |

**Next:** `02_Word_Ladders_Example` builds a graph of 2,442 words and searches
it, and `03_Graphs_Reference` is the lookup guide.